In [ ]:
import requests
import csv
import os

# Base URL
BASE_URL = "https://vzv.nyc/arcgis/rest/services/Vision_Zero"

# List of Vision_Zero services
services = [
    "all_monthly_Injuries_and_fatalities",
    "allFatalities_monthly",
    "allFatalities_yearly",
    "allInjury_yearly",
    "bikeFatalities_and_Injuries_monthly",
    "bikeFatalities_monthly",
    "bikeFatalities_yearly",
    "bikeInjury_yearly",
    "motorFatalities_and_Injuries_monthly",
    "motorFatalities_monthly",
    "motorFatalities_yearly",
    "motorInjury_yearly",
    "OUTREACH",
    "pedFatalities_and_Injuries_monthly",
    "pedFatalities_monthly",
    "pedFatalities_yearly",
    "pedInjury_yearly",
    "SAFETY_INTERVENTIONS",
    "speed_limits",
    "SUMMARY_FATALITIES",
    "SUMMARY_INJURIES"
]

# Max records per request
BATCH_SIZE = 1000

# Create main output folder
os.makedirs('vision_zero_combined_csv', exist_ok=True)

# Fetch available layer IDs for a service
def fetch_layers(service_name):
    url = f"{BASE_URL}/{service_name}/MapServer?f=json"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        layers = data.get('layers', [])
        return [layer['id'] for layer in layers]
    else:
        print(f" Failed to fetch layers for {service_name}")
        return []

# Fetch all available fields for a layer
def fetch_fields(service_name, layer_id):
    url = f"{BASE_URL}/{service_name}/MapServer/{layer_id}?f=json"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        return [field["name"] for field in data.get("fields", [])]
    else:
        print(f" Failed to fetch fields for {service_name} layer {layer_id}")
        return []

# Fetch all data for a layer
def fetch_layer_data(service_name, layer_id, fields):
    all_features = []
    offset = 0

    while True:
        query_url = f"{BASE_URL}/{service_name}/MapServer/{layer_id}/query"
        params = {
            "where": "1=1",
            "outFields": ",".join(fields),
            "f": "json",
            "returnGeometry": True,
            "resultOffset": offset,
            "resultRecordCount": BATCH_SIZE
        }
        response = requests.get(query_url, params=params)
        if response.status_code != 200:
            print(f" Failed to fetch data for {service_name} layer {layer_id}")
            break

        data = response.json()
        features = data.get('features', [])
        
        if not features:
            break
        
        all_features.extend(features)
        offset += BATCH_SIZE

    return all_features

# Main process
for service in services:
    print(f"\n Processing service: {service}")

    all_combined_features = []
    all_fieldnames = set()

    # Fetch all layers for the service
    layers = fetch_layers(service)
    if not layers:
        print(f"⚠️ No layers found for {service}")
        continue

    for layer_id in layers:
        print(f"   Fetching layer {layer_id}...")
        
        # Fetch available fields for this layer
        fields = fetch_fields(service, layer_id)
        if not fields:
            continue

        # Fetch features for this layer
        features = fetch_layer_data(service, layer_id, fields)
        if not features:
            continue

        for feature in features:
            row = feature.get('attributes', {})
            if 'geometry' in feature:
                row['geometry_x'] = feature['geometry'].get('x')
                row['geometry_y'] = feature['geometry'].get('y')
            # Add layer id as extra information
            row['source_layer_id'] = layer_id
            all_combined_features.append(row)
            all_fieldnames.update(row.keys())

    # If no features collected, skip saving
    if not all_combined_features:
        print(f" No data found for {service}")
        continue

    # Save combined data into one CSV file
    output_path = os.path.join('vision_zero_combined_csv', f"{service}.csv")
    with open(output_path, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=sorted(all_fieldnames))
        writer.writeheader()
        for record in all_combined_features:
            writer.writerow(record)

    print(f"Combined {len(all_combined_features)} records into {output_path}")

print("\nAll services processed and saved as combined CSV files!")



 Processing service: SUMMARY_FATALITIES
   Fetching layer 0...
   Fetching layer 1...
   Fetching layer 2...
   Fetching layer 3...
Combined 204 records into vision_zero_combined_csv\SUMMARY_FATALITIES.csv

 Processing service: SUMMARY_INJURIES
   Fetching layer 0...
   Fetching layer 1...
   Fetching layer 2...
   Fetching layer 3...
Combined 204 records into vision_zero_combined_csv\SUMMARY_INJURIES.csv

All services processed and saved as combined CSV files!
